# 06 - LSTM Direction Classifier: Training Pipeline

**Goal:** Train an LSTM to predict whether the next bar closes *higher* (1) or *lower* (0) than the current bar.  
**Data source:** `data/feature_store` (Hive-partitioned Parquet written by the ETL pipeline).  
**Model:** `models.lstm_model.LSTMModel` — stacked LSTM with a single linear head

### Pipeline overview
1. Load raw features from the feature store via `MLDataLoader`
2. Build the next-bar direction target
3. Normalise features (per-feature zero-mean / unit-std, fit on train only)
4. Slide windows → `(n_samples, seq_len, n_features)` arrays
5. Train / validate / test split (time-ordered)
6. Train the LSTM
7. Evaluate: metrics + confusion matrix
8. Plot training curves
9. Save the model

## 0. Setup

In [ ]:
import sys
from pathlib import Path

# Make sure the project root is on the path regardless of where the notebook runs.
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

from financials.etl_pipeline import MLDataLoader
from models.lstm_model import LSTMModel

plt.style.use("seaborn-v0_8-darkgrid")
pd.set_option("display.max_columns", 40)
print("Imports OK")

## 1. Configuration

Adjust `DATASET_PATH`, `SYMBOLS`, and the date range to match the local feature store.

In [ ]:
# ── Data ──────────────────────────────────────────────────────────────────────
DATASET_PATH = PROJECT_ROOT / "data" / "feature_store"

columns_loader = MLDataLoader(DATASET_PATH)
SYMBOLS      = list(columns_loader.available_symbols())
TRAIN_START  = "2020-01-01"
TRAIN_END    = "2021-12-31"
TEST_START   = "2022-01-01"   # held-out test window
TEST_END     = "2022-06-31"

# ── Sequence ──────────────────────────────────────────────────────────────────
SEQUENCE_LENGTH = 60     # look-back window in bars (~3 months of daily data)
STRIDE          = 1      # step between consecutive windows

# ── Model ─────────────────────────────────────────────────────────────────────
MODEL_NAME   = "lstm_direction_v1"
HIDDEN_SIZE  = 128
NUM_LAYERS   = 2
DROPOUT      = 0.2
LR           = 1e-3
BATCH_SIZE   = 256
EPOCHS       = 60
RANDOM_STATE = 42

# ── Feature columns (~30) ─────────────────────────────────────────────────────
# These names match what the ETL pipeline writes to the feature store.
# Series features keep their plain name; composite features are <name>_<sub>.
FEATURE_COLUMNS = [
    # OHLCV (5)
    "open", "high", "low", "close", "volume",
    # Returns (2)
    "returns", "log_returns",
    # Moving averages (4)
    "sma", "ema", "wma", "hma",
    # Momentum (1)
    "rsi",
    # MACD (3)
    "macd_macd", "macd_signal", "macd_histogram",
    # Bollinger Bands (3)
    "bollinger_bands_upper", "bollinger_bands_middle", "bollinger_bands_lower",
    # Volatility / ATR (2)
    "atr", "volatility",
    # Rolling statistics (4)
    "rolling_stats_rolling_mean", "rolling_stats_rolling_std",
    "rolling_stats_rolling_skew", "rolling_stats_rolling_kurtosis",
    # Volume (2)
    "volume_features_volume_sma", "volume_features_volume_ratio", "volume_features_obv",
    # Price structure (3)
    "price_features_high_low_range",
    "price_features_close_open_range",
    "price_features_gap",
]

N_FEATURES = len(FEATURE_COLUMNS)
print(f"Feature columns: {N_FEATURES}")
print(FEATURE_COLUMNS)

## 2. Load, normalise, and build sequences

All three steps run in a single two-pass loop over symbols so that peak RAM is bounded by one ticker's rows, not the full dataset.

In [ ]:
loader = MLDataLoader(DATASET_PATH)

print("Available dates (first/last):",
      loader.available_dates()[:1], "...", loader.available_dates()[-1:])
print("Available symbols:", loader.available_symbols())

In [ ]:
load_cols = list(dict.fromkeys(["timestamp", "symbol", "close"] + FEATURE_COLUMNS))


def add_direction_target(df: pd.DataFrame) -> pd.DataFrame:
    """Add a `direction` column: 1 if the NEXT bar's close > current close."""
    df = df.copy()
    df["direction"] = (
        df.groupby("symbol")["close"]
        .transform(lambda s: (s.shift(-1) > s).astype("Int8"))
    )
    df = df.dropna(subset=["direction"])
    df["direction"] = df["direction"].astype(int)
    return df


# Pass 1: fit scaler on training data, one symbol at a time
# Peak RAM = one symbol's rows, not the entire training set.
scaler = StandardScaler()
for sym in SYMBOLS:
    df_sym = loader.load_pandas(
        symbols=[sym], start=TRAIN_START, end=TRAIN_END, columns=load_cols
    ).dropna(subset=FEATURE_COLUMNS)
    if len(df_sym) > 0:
        scaler.partial_fit(df_sym[FEATURE_COLUMNS])

print("Scaler fitted.")


# Pass 2: build sequences, one symbol at a time
def _sequences_for_split(split_start: str, split_end: str):
    X_list, y_list = [], []
    for sym in SYMBOLS:
        df_sym = (
            loader.load_pandas(
                symbols=[sym], start=split_start, end=split_end, columns=load_cols
            )
            .dropna(subset=FEATURE_COLUMNS)
            .reset_index(drop=True)
            .pipe(add_direction_target)
            .dropna(subset=["direction"])
            .reset_index(drop=True)
        )
        if len(df_sym) <= SEQUENCE_LENGTH:
            continue
        feats  = scaler.transform(df_sym[FEATURE_COLUMNS].values).astype("float32")
        labels = df_sym["direction"].to_numpy(dtype="float32")
        for i in range(SEQUENCE_LENGTH, len(df_sym), STRIDE):
            X_list.append(feats[i - SEQUENCE_LENGTH : i])
            y_list.append(labels[i - 1])
    return np.stack(X_list), np.array(y_list, dtype="float32")


print("Building training sequences...")
X_train, y_train = _sequences_for_split(TRAIN_START, TRAIN_END)

print("Building test sequences...")
X_test, y_test = _sequences_for_split(TEST_START, TEST_END)

print(f"X_train: {X_train.shape}   y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape}   y_test:  {y_test.shape}")
print(f"Train class balance:  up={y_train.mean():.1%}   down={(1-y_train.mean()):.1%}")
print(f"Test  class balance:  up={y_test.mean():.1%}   down={(1-y_test.mean()):.1%}")


## 3. Feature exploration

Load one representative symbol for NaN checks and distribution plots.  
The full dataset is consumed as a stream during loading (above).

In [ ]:
_sample_sym = SYMBOLS[0]
df_sample = (
    loader.load_pandas(
        symbols=[_sample_sym], start=TRAIN_START, end=TRAIN_END, columns=load_cols
    )
    .dropna(subset=FEATURE_COLUMNS)
    .reset_index(drop=True)
)

print(f"Sample symbol: {_sample_sym}  ({len(df_sample):,} rows)")
print("NaN counts per feature (sample):")
nan_counts = df_sample[FEATURE_COLUMNS].isna().sum()
print(nan_counts[nan_counts > 0].to_string() or "  None - all features present")


In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(16, 12))
for ax, col in zip(axes.ravel(), FEATURE_COLUMNS[:16]):
    df_sample[col].dropna().hist(bins=40, ax=ax, color="steelblue", edgecolor="none")
    ax.set_title(col, fontsize=8)
    ax.tick_params(labelsize=7)
plt.suptitle(f"Feature distributions ({_sample_sym} train set, first 16 features)", y=1.01)
plt.tight_layout()
plt.show()


## 4. Train the LSTM

In [ ]:
model = LSTMModel(
    name=MODEL_NAME,
    n_features=N_FEATURES,
    sequence_length=SEQUENCE_LENGTH,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    learning_rate=LR,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    random_state=RANDOM_STATE,
)

print(f"Device: {model.device}")
print(f"Parameters: {sum(p.numel() for p in model._net.parameters()):,}")

In [ ]:
# val_split carves out the last 10 % of *training* data for validation
# monitoring (time-ordered, no shuffle).
model.train(X_train, y_train, val_split=0.1)

## 5. Training curves

In [ ]:
history = pd.DataFrame(model.train_history)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(history["epoch"], history["train_loss"], label="train loss")
ax1.plot(history["epoch"], history["val_loss"],   label="val loss", linestyle="--")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("BCE Loss")
ax1.set_title("Loss")
ax1.legend()

ax2.plot(history["epoch"], history["val_acc"], color="darkorange", label="val accuracy")
ax2.axhline(0.5, color="grey", linestyle=":", label="random baseline")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_title("Validation Accuracy")
ax2.legend()

plt.suptitle(f"{MODEL_NAME} — training history", y=1.02)
plt.tight_layout()
plt.show()

print(f"Best val accuracy: {history['val_acc'].max():.4f}  "
      f"(epoch {history['val_acc'].idxmax() + 1})")

## 6. Evaluate on the held-out test set

In [ ]:
y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("=" * 55)
print(f"Test samples : {len(y_test):,}")
print("=" * 55)
print(classification_report(y_test, y_pred, target_names=["Down (0)", "Up (1)"]))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues", ax=ax,
    xticklabels=["Pred Down", "Pred Up"],
    yticklabels=["True Down", "True Up"],
)
ax.set_title(f"{MODEL_NAME} — confusion matrix (test set)")
plt.tight_layout()
plt.show()

In [ ]:
# BaseModel.evaluate() for a uniform metrics dict (used by the registry)
# We pass X_test as a 3-D array; evaluate() calls predict() internally.
metrics = model.evaluate(X_test, pd.Series(y_test, dtype=int))

for key, val in metrics.items():
    if isinstance(val, float):
        print(f"  {key:<20}: {val:.4f}")

## 7. Probability calibration check

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(y_proba[y_test == 0], bins=40, alpha=0.55, label="True Down",  color="tomato")
ax.hist(y_proba[y_test == 1], bins=40, alpha=0.55, label="True Up",    color="steelblue")
ax.axvline(0.5, color="black", linestyle="--", linewidth=1)
ax.set_xlabel("P(Up)")
ax.set_ylabel("Count")
ax.set_title("Predicted probability distribution by true class")
ax.legend()
plt.tight_layout()
plt.show()

## 8. Save the model

In [ ]:
import joblib

MODELS_DIR = PROJECT_ROOT / "models" / "saved"
model_path  = str(MODELS_DIR / f"{MODEL_NAME}.pt")
scaler_path = str(MODELS_DIR / f"{MODEL_NAME}_scaler.joblib")

model.save(model_path)
joblib.dump(scaler, scaler_path)

print(f"Model  saved: {model_path}")
print(f"Scaler saved: {scaler_path}")

## 9. Round-trip load check

In [ ]:
loaded_model = LSTMModel.load(model_path)
y_pred_reload = loaded_model.predict(X_test)

assert np.array_equal(y_pred, y_pred_reload), "Predictions differ after reload!"
print("Round-trip check passed — predictions are identical after save/load.")

## 10. (Optional) Register with ModelRegistry

Uncomment if you have a running PostgreSQL instance and want to persist metadata.

In [ ]:
# from models.registry import ModelRegistry
#
# registry = ModelRegistry()
# training_info = {
#     "symbols"          : SYMBOLS,
#     "features"         : FEATURE_COLUMNS,
#     "sequence_length"  : SEQUENCE_LENGTH,
#     "train_start"      : TRAIN_START,
#     "train_end"        : TRAIN_END,
#     "test_start"       : TEST_START,
#     "test_end"         : TEST_END,
#     "train_samples"    : int(len(X_train)),
#     "test_samples"     : int(len(X_test)),
#     "scaler_path"      : scaler_path,
# }
# registry.register(model, training_info, metrics, description="LSTM direction classifier v1")